In [19]:
from datetime import datetime
from pathlib import Path
import importlib
import math

student_id = "850006642"
data_file = Path.cwd() / f"LHMS_{student_id}.txt"


"""
The two dictionaries below are the main working storage for the application.
rooms keeps room records by room number.
allocations keeps active bookings by room number so that one room cannot be double-booked.
"""
rooms = {}
allocations = {}
billing_history = []


print(data_file.name)

LHMS_850006642.txt


In [20]:
def clean_room_number(room_number):
    """Return a clean room number after checking that it contains digits only."""
    room_number = str(room_number).strip()
    if room_number == "":
        raise ValueError("Room number cannot be blank.")
    if not room_number.isdigit():
        raise ValueError("Room number must contain digits only.")
    return room_number


def clean_positive_money(amount, field_name="amount"):
    """Convert a value into a positive float for prices and rates."""
    try:
        amount = float(amount)
    except ValueError:
        raise ValueError(f"{field_name} must be a number.")

    if amount <= 0:
        raise ValueError(f"{field_name} must be greater than zero.")
    return amount


def clean_positive_integer(value, field_name="number"):
    """Convert a value into a positive integer."""
    try:
        value = int(value)
    except ValueError:
        raise ValueError(f"{field_name} must be a whole number.")

    if value <= 0:
        raise ValueError(f"{field_name} must be greater than zero.")
    return value


def money(value):
    """Format a number as dollars for the bill output."""
    return f"${value:.2f}"


def make_booking_id(room_number):
    """Create a simple booking ID using the room number and current time."""
    return "BK" + room_number + datetime.now().strftime("%Y%m%d%H%M%S")


def get_room(room_number):
    """Find a room by room number after validating the value."""
    room_number = clean_room_number(room_number)
    return rooms.get(room_number)

## Safe input functions for the menu


In [21]:
def ask_text(prompt):
    while True:
        try:
            value = input(prompt).strip()
            if value == "":
                print("Please enter a value. It cannot be blank.")
            else:
                return value
        except EOFError:
            print("Input ended before a value was entered.")
            raise


def ask_integer(prompt):
    while True:
        try:
            return clean_positive_integer(input(prompt), "Value")
        except ValueError as error:
            print(error)
        except EOFError:
            print("Input ended before a number was entered.")
            raise


def ask_money(prompt):
    while True:
        try:
            return clean_positive_money(input(prompt), "Rate")
        except ValueError as error:
            print(error)
        except EOFError:
            print("Input ended before a rate was entered.")
            raise

## Menu option 1: Add room


In [29]:
def add_room(room_number, room_type, nightly_rate):
    try:
        room_number = clean_room_number(room_number)
        room_type = str(room_type).strip()
        
        rate = clean_positive_money(nightly_rate, "rate")

        if room_type == "":
            return "Room type cannot be blank."
        
        if room_number in rooms:
            return f"Room {room_number} already exists."

        rooms[room_number] = {
            "room_number": room_number,
            "room_type": room_type,
            "rate": rate,
            
            "status": "Available"
        }
        return f"Room {room_number} has been added."

    except ValueError as error:
        return f"Room was not added. {error}"
    except TypeError as error:
        return f"Room was not added because of a data type problem: {error}"


def menu_add_room():
    print("Add Room")
    room_number = ask_text("Room number: ")
    room_type = ask_text("Room type: ")
    rate = ask_money("rate: ")
    
    print(add_room(room_number, room_type, rate))

## Menu option 2: Delete room

A room can only be deleted if it exists and is not currently booked.

In [23]:
def delete_room(room_number):
    try:
        room_number = clean_room_number(room_number)

        if room_number not in rooms:
            return f"Room {room_number} does not exist."
        if room_number in allocations:
            return f"Room {room_number} is currently booked, so it cannot be deleted."

        del rooms[room_number]
        return f"Room {room_number} has been deleted."

    except ValueError as error:
        return f"Room was not deleted. {error}"


def menu_delete_room():
    print("Delete Room")
    room_number = ask_text("Room number to delete: ")
    print(delete_room(room_number))

## Menu option 3: Display room details

This option shows all room information before booking. It also shows whether each room is available or occupied.

In [24]:
def display_room_details():
    if len(rooms) == 0:
        print("No rooms have been added yet.")
        return

    print("Room Details")
    for room_number in sorted(rooms):
        try:
            room = rooms[room_number]
            print(f"Room {room['room_number']}")
            print(f"Type: {room['room_type']}")
            print(f"Rate: {money(room['rate'])}")
            
            print(f"Status: {room['status']}")
            print()
        except KeyError as error:
            print("A room record is missing this field:", error)


def menu_display_rooms():
    display_room_details()

## Menu option 4: Allocate room

This option books an available room for a customer. It checks the room number, customer details, and number of nights before creating the booking.

In [25]:
def allocate_room(room_number, customer_name, phone_number, nights):
    try:
        room_number = clean_room_number(room_number)
        customer_name = str(customer_name).strip()
        phone_number = str(phone_number).strip()
        nights = clean_positive_integer(nights, "Number of nights")

        if room_number not in rooms:
            return f"Room {room_number} cannot be booked because it does not exist."
        if rooms[room_number]["status"] != "Available":
            return f"Room {room_number} is not available."
        if customer_name == "":
            return "Customer name cannot be blank."
        if phone_number == "":
            return "Phone number cannot be blank."

        booking = {
            "booking_id": make_booking_id(room_number),
            "room_number": room_number,
            "customer_name": customer_name,
            "phone_number": phone_number,
            "nights": nights,
            "booked_at": datetime.now().strftime("%d/%m/%Y %I:%M %p")
        }
        allocations[room_number] = booking
        rooms[room_number]["status"] = "Occupied"
        return f"Room {room_number} has been allocated to {customer_name}. Booking ID: {booking['booking_id']}"

    except ValueError as error:
        return f"Room was not allocated. {error}"
    except TypeError as error:
        return f"Room was not allocated because of a data type problem: {error}"
    except NameError as error:
        return f"Room was not allocated because a variable name is incorrect: {error}"


def menu_allocate_room():
    print("Allocate Room")
    room_number = ask_text("Room number: ")
    customer_name = ask_text("Customer name: ")
    phone_number = ask_text("Phone number: ")
    nights = ask_integer("Number of nights: ")
    print(allocate_room(room_number, customer_name, phone_number, nights))

## Menu option 5: Display room allocation details

This option shows the current active room bookings.

In [26]:
def display_room_allocation_details():
    if len(allocations) == 0:
        print("No rooms are currently allocated.")
        return

    print("Current Room Allocations")
    for room_number in sorted(allocations):
        try:
            booking = allocations[room_number]
            room = rooms[room_number]
            print(f"Booking ID: {booking['booking_id']}")
            print(f"Room: {booking['room_number']} ({room['room_type']})")
            print(f"Customer: {booking['customer_name']}")
            print(f"Phone: {booking['phone_number']}")
            print(f"Nights: {booking['nights']}")
            print(f"Rate: {money(room['rate'])} per night")
            print(f"Booked at: {booking['booked_at']}")
            print()
        except KeyError as error:
            print("An allocation record is missing this field:", error)


def menu_display_allocations():
    display_room_allocation_details()

## Menu option 6: Billing and de-allocation

This option creates the bill for the booked room and then makes the room available again.

In [27]:
def prepare_bill(room_number):
    room_number = clean_room_number(room_number)
    if room_number not in allocations:
        raise IndexError(f"No active booking was found for room {room_number}.")

    booking = allocations[room_number]
    room = rooms[room_number]
    subtotal = room["rate"] * booking["nights"]
    
    total = subtotal

    return {
        "booking_id": booking["booking_id"],
        "room_number": room_number,
        "customer_name": booking["customer_name"],
        "nights": booking["nights"],
        "rate": room["rate"],
        "subtotal": subtotal,
        
        "total": total,
        "checked_out_at": datetime.now().strftime("%d/%m/%Y %I:%M %p")
    }


def billing_and_deallocate(room_number):
    try:
        room_number = clean_room_number(room_number)
        bill = prepare_bill(room_number)

        del allocations[room_number]
        rooms[room_number]["status"] = "Available"
        billing_history.append(bill)

        print("Billing and De-allocation")
        print(f"Booking ID: {bill['booking_id']}")
        print(f"Customer: {bill['customer_name']}")
        print(f"Room: {bill['room_number']}")
        print(f"Nights: {bill['nights']}")
        print(f"Rate: {money(bill['rate'])}")
        print(f"Subtotal: {money(bill['subtotal'])}")
        
        print(f"Total amount: {money(bill['total'])}")
        print(f"Room {room_number} is now available.")

    except ValueError as error:
        print("Billing was not completed.", error)
    except IndexError as error:
        print("Billing was not completed.", error)
    except ZeroDivisionError:
        print("Billing was not completed because the calculation used zero incorrectly.")


def menu_billing_and_deallocate():
    print("Billing and De-allocation")
    room_number = ask_text("Room number for checkout: ")
    billing_and_deallocate(room_number)

## Menu option 7: Save room allocation to a file

This option creates `LHMS_StudentID.txt` in the notebook folder and writes the current allocation information into it.

In [15]:
def build_allocation_report():
    lines = []
    lines.append("Langham Hotel Management System")
    lines.append("Current Room Allocation Report")
    lines.append("Created: " + datetime.now().strftime("%d/%m/%Y %I:%M %p"))
    lines.append("")

    if len(allocations) == 0:
        lines.append("There are no active room allocations.")
    else:
        for room_number in sorted(allocations):
            booking = allocations[room_number]
            room = rooms[room_number]
            lines.append(f"Booking ID: {booking['booking_id']}")
            lines.append(f"Room: {booking['room_number']}")
            lines.append(f"Room type: {room['room_type']}")
            lines.append(f"Customer: {booking['customer_name']}")
            lines.append(f"Phone: {booking['phone_number']}")
            lines.append(f"Nights: {booking['nights']}")
            lines.append(f"Rate: {money(room['rate'])}")
            lines.append(f"Booked at: {booking['booked_at']}")
            lines.append("")

    return "\n".join(lines)


def save_room_allocation_to_file():
    try:
        report = build_allocation_report()
        with open(data_file, "w", encoding="utf-8") as file:
            file.write(report)
        print("Room allocation data has been saved to", data_file.name)
    except IOError as error:
        print("The room allocation file could not be saved.", error)
    except OSError as error:
        print("A file system error happened while saving.", error)


def menu_save_to_file():
    print("Save Room Allocation to File")
    save_room_allocation_to_file()

## Menu option 8: Display room allocation from the file

This option reads the saved text file and displays its contents in the notebook output.

In [16]:
def display_room_allocation_from_file():
    try:
        with open(data_file, "r", encoding="utf-8") as file:
            content = file.read()

        print("Room Allocation File Content")
        if content.strip() == "":
            print("The allocation file is empty.")
        else:
            print(content)

    except FileNotFoundError:
        print("The allocation file does not exist yet. Use menu option 7 first.")
    except IOError as error:
        print("The allocation file could not be opened.", error)


def menu_display_file():
    print("Display Room Allocation from File")
    display_room_allocation_from_file()

## Menu option 9: Backup and clear room allocation file

This option copies the saved allocation file into a timestamped backup file and then clears the original file.

In [17]:
def backup_and_clear_room_allocation_file():
    try:
        if not data_file.exists():
            print("The allocation file does not exist yet. Use menu option 7 first.")
            return

        with open(data_file, "r", encoding="utf-8") as file:
            content = file.read()

        if content.strip() == "":
            print("The allocation file is already empty. No backup was created.")
            return

        time_text = datetime.now().strftime("%Y%m%d_%H%M%S")
        backup_file = Path.cwd() / f"LHMS_{student_id}_Backup_{time_text}.txt"

        with open(backup_file, "a", encoding="utf-8") as file:
            file.write(content)
            file.write("\n")

        with open(data_file, "w", encoding="utf-8") as file:
            file.write("")

        print("Backup file created:", backup_file.name)
        print("The original allocation file has been cleared.")

    except FileNotFoundError:
        print("The allocation file was not found.")
    except IOError as error:
        print("Backup or clear operation failed.", error)


def menu_backup_and_clear():
    print("Backup and Clear Room Allocation File")
    backup_and_clear_room_allocation_file()

## Main application menu

Run `run_application()` when you want to use the program interactively. The test cell later in the notebook calls the functions directly, so it does not need keyboard input.

In [32]:
def show_menu():
    print()
    print("Langham Hotel Management System")
    print("1. Add Room")
    print("2. Delete Room")
    print("3. Display Room Details")
    print("4. Allocate Room")
    print("5. Display Room Allocation Details")
    print("6. Billing and De-allocation")
    print("7. Save Room Allocation to File")
    print("8. Display Room Allocation from File")
    print("9. Backup and Clear Room Allocation File")
    print("0. Exit Application")


def run_application():
    print("Welcome to the Langham Hotel Management System.")

    while True:
        try:
            show_menu()
            choice = input("Enter your choice from 0 to 9: ").strip()

            if choice == "1":
                menu_add_room()
            elif choice == "2":
                menu_delete_room()
            elif choice == "3":
                menu_display_rooms()
            elif choice == "4":
                menu_allocate_room()
            elif choice == "5":
                menu_display_allocations()
            elif choice == "6":
                menu_billing_and_deallocate()
            elif choice == "7":
                menu_save_to_file()
            elif choice == "8":
                menu_display_file()
            elif choice == "9":
                menu_backup_and_clear()
            elif choice == "0":
                print("Thank you. The application is now closed.")
                break
            else:
                print("Please choose a number from 0 to 9.")

        except KeyboardInterrupt:
            print("The program was stopped from the keyboard.")
            break
        except EOFError:
            print("No more input is available. The program will now stop.")
            break
        except Exception as error:
            print("An unexpected error was handled:", error)


run_application()

Welcome to the Langham Hotel Management System.

Langham Hotel Management System
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Room
5. Display Room Allocation Details
6. Billing and De-allocation
7. Save Room Allocation to File
8. Display Room Allocation from File
9. Backup and Clear Room Allocation File
0. Exit Application


Enter your choice from 0 to 9:  1


Add Room


Room number:  101
Room type:  good
rate:  80


Room 101 has been added.

Langham Hotel Management System
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Room
5. Display Room Allocation Details
6. Billing and De-allocation
7. Save Room Allocation to File
8. Display Room Allocation from File
9. Backup and Clear Room Allocation File
0. Exit Application


Enter your choice from 0 to 9:  3


Room Details
Room 101
Type: good
Rate: $80.00
Status: Available


Langham Hotel Management System
1. Add Room
2. Delete Room
3. Display Room Details
4. Allocate Room
5. Display Room Allocation Details
6. Billing and De-allocation
7. Save Room Allocation to File
8. Display Room Allocation from File
9. Backup and Clear Room Allocation File
0. Exit Application


Enter your choice from 0 to 9:  0


Thank you. The application is now closed.


## Test run for screenshots

The next cell runs the main operations with sample data. It is useful for checking that the application works before using the interactive menu.

In [14]:
rooms.clear()
allocations.clear()
billing_history.clear()

print("Full system test run")
print()

print("Option 1: Add rooms")
print(add_room("101", "Standard", 120, "Queen bed, WiFi, TV"))
print(add_room("204", "Deluxe", 185, "King bed, WiFi, city view"))
print(add_room("305", "Suite", 310, "Lounge, breakfast, harbour view"))
print(add_room("101", "Standard", 120, "Duplicate check"))
print()

print("Option 3: Display rooms")
display_room_details()

print("Option 4: Allocate rooms")
print(allocate_room("204", "Mia Thompson", "0215554433", 3))
print(allocate_room("305", "Daniel Lee", "027991100", 2))
print(allocate_room("204", "Second Guest", "020000000", 1))
print()

print("Option 5: Display allocation details")
display_room_allocation_details()

print("Option 2: Delete room")
print(delete_room("101"))
print(delete_room("204"))
print()

print("Option 7: Save allocation to file")
save_room_allocation_to_file()
print()

print("Option 8: Display allocation from file")
display_room_allocation_from_file()
print()

print("Option 9: Backup and clear allocation file")
backup_and_clear_room_allocation_file()
print()

print("Option 6: Billing and de-allocation")
billing_and_deallocate("204")
print()

print("Option 5 after checkout")
display_room_allocation_details()
print()

print("Option 0: Exit application")
print("Thank you. The application is now closed.")

Full system test run

Option 1: Add rooms
Room 101 has been added.
Room 204 has been added.
Room 305 has been added.
Room 101 already exists.

Option 3: Display rooms
Room Details
Room 101
Type: Standard
Rate per night: $120.00
Features: Queen bed, WiFi, TV
Status: Available

Room 204
Type: Deluxe
Rate per night: $185.00
Features: King bed, WiFi, city view
Status: Available

Room 305
Type: Suite
Rate per night: $310.00
Features: Lounge, breakfast, harbour view
Status: Available

Option 4: Allocate rooms
Room 204 has been allocated to Mia Thompson. Booking ID: BK20420260611200331
Room 305 has been allocated to Daniel Lee. Booking ID: BK30520260611200331
Room 204 is not available.

Option 5: Display allocation details
Current Room Allocations
Booking ID: BK20420260611200331
Room: 204 (Deluxe)
Customer: Mia Thompson
Phone: 0215554433
Nights: 3
Rate: $185.00 per night
Booked at: 11/06/2026 08:03 PM

Booking ID: BK30520260611200331
Room: 305 (Suite)
Customer: Daniel Lee
Phone: 027991100
Nig

## Task 2: Six errors detected and corrected

The following development notes explain six errors that were considered while making the program more reliable.

In [15]:
fixed_errors = [
    {
        "error": "ValueError from room number input",
        "problem": "Letters or blank text could be entered instead of a room number.",
        "fix": "clean_room_number() now checks for blank values and digits only."
    },
    {
        "error": "Logical error from duplicate rooms",
        "problem": "The same room number could be added more than once.",
        "fix": "add_room() checks whether the room number already exists before saving it."
    },
    {
        "error": "Logical error from double booking",
        "problem": "An occupied room could be allocated again if the status was not checked.",
        "fix": "allocate_room() checks that the room status is Available before booking."
    },
    {
        "error": "IndexError during checkout",
        "problem": "Billing could be requested for a room that did not have an active booking.",
        "fix": "prepare_bill() raises and handles a clear IndexError for missing bookings."
    },
    {
        "error": "FileNotFoundError when reading saved data",
        "problem": "The program could try to read the allocation file before it was created.",
        "fix": "display_room_allocation_from_file() catches FileNotFoundError and explains option 7 should be used first."
    },
    {
        "error": "IOError during file work",
        "problem": "Saving, reading, backing up, or clearing files can fail because of file system issues.",
        "fix": "The file operations use try and except blocks and close files automatically with with open()."
    }
]

print("Error correction notes")
for number, item in enumerate(fixed_errors, start=1):
    print(f"{number}. {item['error']}")
    print("Problem:", item["problem"])
    print("Fix:", item["fix"])
    print()

Error correction notes
1. ValueError from room number input
Problem: Letters or blank text could be entered instead of a room number.
Fix: clean_room_number() now checks for blank values and digits only.

2. Logical error from duplicate rooms
Problem: The same room number could be added more than once.
Fix: add_room() checks whether the room number already exists before saving it.

3. Logical error from double booking
Problem: An occupied room could be allocated again if the status was not checked.
Fix: allocate_room() checks that the room status is Available before booking.

4. IndexError during checkout
Problem: Billing could be requested for a room that did not have an active booking.
Fix: prepare_bill() raises and handles a clear IndexError for missing bookings.

5. FileNotFoundError when reading saved data
Problem: The program could try to read the allocation file before it was created.
Fix: display_room_allocation_from_file() catches FileNotFoundError and explains option 7 should

## Task 2: Exception handling demonstration

This cell safely demonstrates all eleven exception types listed in the assessment. Each error is intentionally triggered inside a `try` block and caught by the correct `except` block.

In [16]:
def demonstrate_required_exceptions():
    print("Exception handling demonstration")

    try:
        compile("if True print('missing colon')", "demo", "exec")
    except SyntaxError as error:
        print("1. SyntaxError caught:", error.__class__.__name__)

    try:
        int("three")
    except ValueError as error:
        print("2. ValueError caught:", error.__class__.__name__)

    try:
        120 / 0
    except ZeroDivisionError as error:
        print("3. ZeroDivisionError caught:", error.__class__.__name__)

    try:
        sample_rooms = ["101", "204"]
        sample_rooms[5]
    except IndexError as error:
        print("4. IndexError caught:", error.__class__.__name__)

    try:
        not_created_room_name
    except NameError as error:
        print("5. NameError caught:", error.__class__.__name__)

    try:
        "Room" + 204
    except TypeError as error:
        print("6. TypeError caught:", error.__class__.__name__)

    try:
        math.exp(1000)
    except OverflowError as error:
        print("7. OverflowError caught:", error.__class__.__name__)

    try:
        with open(Path.cwd(), "r", encoding="utf-8") as file:
            file.read()
    except IOError as error:
        print("8. IOError caught:", error.__class__.__name__)

    try:
        importlib.import_module("module_that_does_not_exist_for_lhms")
    except ImportError as error:
        print("9. ImportError caught:", error.__class__.__name__)

    try:
        raise EOFError("Input ended unexpectedly during a menu entry.")
    except EOFError as error:
        print("10. EOFError caught:", error.__class__.__name__)

    try:
        with open("missing_lhms_file.txt", "r", encoding="utf-8") as file:
            file.read()
    except FileNotFoundError as error:
        print("11. FileNotFoundError caught:", error.__class__.__name__)


demonstrate_required_exceptions()

Exception handling demonstration
1. SyntaxError caught: SyntaxError
2. ValueError caught: ValueError
3. ZeroDivisionError caught: ZeroDivisionError
4. IndexError caught: IndexError
5. NameError caught: NameError
6. TypeError caught: TypeError
7. OverflowError caught: OverflowError
8. IOError caught: IsADirectoryError
9. ImportError caught: ModuleNotFoundError
10. EOFError caught: EOFError
11. FileNotFoundError caught: FileNotFoundError


## Pseudocode

The pseudocode below keeps one statement per line, uses capitalised keywords, uses indentation to show structure, ends multiline structures, and avoids Python-specific syntax.

```text
START
    SET rooms TO empty collection
    SET allocations TO empty collection
    SET billing_history TO empty collection
    SET data_file TO LHMS_StudentID text file

    WRITE welcome message

    WHILE user has not selected Exit
        DISPLAY menu options 0 to 9
        READ user_choice

        IF user_choice is Add Room THEN
            READ room_number
            READ room_type
            READ rate
            
            IF room_number is invalid THEN
                WRITE error message
            ELSE IF room already exists THEN
                WRITE duplicate room message
            ELSE
                SAVE room as Available
                WRITE success message
            ENDIF
        ENDIF

        IF user_choice is Delete Room THEN
            READ room_number
            IF room does not exist THEN
                WRITE room not found message
            ELSE IF room is allocated THEN
                WRITE room cannot be deleted message
            ELSE
                DELETE room
                WRITE success message
            ENDIF
        ENDIF

        IF user_choice is Display Room Details THEN
            IF no rooms exist THEN
                WRITE no room message
            ELSE
                FOR each room in rooms
                    WRITE room number, type, rate and status
                ENDFOR
            ENDIF
        ENDIF

        IF user_choice is Allocate Room THEN
            READ room_number
            READ customer_name
            READ phone_number
            READ number_of_nights
            IF room does not exist THEN
                WRITE room not found message
            ELSE IF room is not available THEN
                WRITE room unavailable message
            ELSE
                CREATE booking record
                CHANGE room status to Occupied
                WRITE booking confirmation
            ENDIF
        ENDIF

        IF user_choice is Display Room Allocation Details THEN
            IF no allocations exist THEN
                WRITE no allocation message
            ELSE
                FOR each allocation in allocations
                    WRITE booking details
                ENDFOR
            ENDIF
        ENDIF

        IF user_choice is Billing and De-allocation THEN
            READ room_number
            IF no active booking exists THEN
                WRITE no booking message
            ELSE
                CALCULATE subtotal
                
                CALCULATE total
                WRITE bill
                REMOVE allocation
                CHANGE room status to Available
            ENDIF
        ENDIF

        IF user_choice is Save Room Allocation to File THEN
            CREATE allocation report
            WRITE report to LHMS_StudentID text file
            CLOSE file
            WRITE saved message
        ENDIF

        IF user_choice is Display Room Allocation from File THEN
            OPEN allocation file
            READ file content
            WRITE file content
            CLOSE file
        ENDIF

        IF user_choice is Backup and Clear Room Allocation File THEN
            OPEN allocation file
            READ file content
            APPEND content to timestamped backup file
            CLEAR original allocation file
            CLOSE files
            WRITE backup message
        ENDIF

        IF user_choice is Exit THEN
            WRITE closing message
            STOP loop
        ENDIF
    ENDWHILE
END
```

## Functions and built-in modules used

- `datetime.now()` is used for booking IDs, report dates, and backup timestamps.
- `Path.cwd()` is used to save the text file in the same working folder as the notebook.
- `open()` is used for file writing, reading, backup, and clearing.
- `sorted()` is used to show rooms and allocations in room-number order.
- `str()`, `int()`, and `float()` are used to validate user input.
- User-defined functions such as `add_room()`, `delete_room()`, `allocate_room()`, `billing_and_deallocate()`, and `backup_and_clear_room_allocation_file()` keep the application organised.

